# Análise Exploratória das Métricas Locais de Conectividade

Este notebook apresenta a análise exploratória das métricas locais (por nó)
extraídas das redes funcionais cerebrais obtidas a partir de sinais de EEG
em estado de repouso.

Diferentemente das métricas globais, as métricas locais permitem investigar
a heterogeneidade topológica entre diferentes regiões cerebrais, sendo
potencialmente mais informativas para tarefas de caracterização de grupos
e aprendizado de máquina.

Objetivos desta etapa:
- analisar a distribuição das métricas locais por nó;
- investigar diferenças entre condições experimentais (EO vs EC);
- avaliar a variabilidade intersujeitos;
- preparar as features para a etapa de redução de dimensionalidade (PCA).


In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"

### Carregando o dataset

In [2]:
DATASET_PATH = "../../data/processed/dataset_features.parquet"

df = pd.read_parquet(DATASET_PATH)

df.shape

(108, 360)

In [3]:
df

,degree_AF3,clustering_AF3,betweenness_AF3,hub_frequency_AF3,sd_degree_AF3,bin_clustering_AF3,degree_AF4,clustering_AF4,betweenness_AF4,hub_frequency_AF4,...,age_60-65,age_65-70,age_70-75,age_75-80,gender_2,handedness_left,handedness_right,education_gymnasium,education_realschule,relationship_status_yes
0,0.403697,0.356960,0.005195,0.001017,0.153145,0.624913,0.418594,0.369780,0.000000,0.002542,...,False,False,False,False,True,False,True,True,False,False
1,0.454580,0.415602,0.003759,0.000000,0.159919,0.679399,0.458031,0.412871,0.008772,0.004067,...,False,False,False,False,True,False,True,True,False,False
2,0.588981,0.575718,0.000584,0.006609,0.193803,0.711421,0.571515,0.562412,0.000000,0.001525,...,False,False,False,False,True,False,True,False,True,True
3,0.454194,0.443422,0.000565,0.000508,0.180212,0.657361,0.445602,0.438874,0.000000,0.000508,...,False,False,False,False,True,False,True,False,True,True
4,0.643513,0.612853,0.000627,0.002542,0.155180,0.757100,0.643611,0.612303,0.001880,0.000000,...,False,False,False,False,False,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,0.395055,0.393771,0.002020,0.001017,0.162625,0.622700,0.404030,0.397058,0.016835,0.000000,...,False,False,False,False,True,True,False,True,False,True
104,0.474781,0.484025,0.000000,0.001017,0.194391,0.705043,0.365710,0.409217,0.000000,0.000000,...,False,False,False,False,True,False,True,True,False,False
105,0.419353,0.433576,0.008475,0.000000,0.198661,0.660947,0.362828,0.382282,0.001695,0.001017,...,False,False,False,False,True,False,True,True,False,False
106,0.581148,0.566322,0.011105,0.001017,0.193871,0.732719,0.566560,0.559494,0.001753,0.000000,...,False,False,False,False,True,False,True,True,False,True


### métricas globais × métricas locais

In [4]:
GLOBAL_FEATURES = [
    "edges_mean",
    "edges_std",
    "edges_cv"
]

NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "hub_frequency_",
]

In [5]:
NODE_FEATURES = [
    c for c in df.columns
    if any(c.startswith(p) for p in NODE_FEATURE_PREFIXES)
]

In [6]:
len(NODE_FEATURES)

212

DataFrame apenas com métricas locais

In [7]:
df_node = df[["subject_id", "condition"] + NODE_FEATURES].copy()
df_node.shape

(108, 214)

Distribuição de Métricas Locais

In [8]:
degree_cols = [c for c in NODE_FEATURES if c.startswith("degree_")]

df_degree_long = df_node.melt(
    id_vars=["subject_id", "condition"],
    value_vars=degree_cols,
    var_name="node",
    value_name="degree"
)

In [9]:
df_degree_long

,subject_id,condition,node,degree
0,sub-010069,EC,degree_AF3,0.403697
1,sub-010069,EO,degree_AF3,0.454580
2,sub-010070,EC,degree_AF3,0.588981
3,sub-010070,EO,degree_AF3,0.454194
4,sub-010079,EC,degree_AF3,0.643513
...,...,...,...,...
5719,sub-010315,EO,degree_TP8,0.530896
5720,sub-010316,EC,degree_TP8,NaN
5721,sub-010316,EO,degree_TP8,0.633689
5722,sub-010318,EC,degree_TP8,0.660147


In [10]:
fig = px.box(
    df_degree_long,
    x="node",
    y="degree",
    color="condition",
    title="Distribuição do Grau por Nó e Condição",
    points=False
)
fig.update_layout(xaxis_tickangle=45)
fig.show()

### Variabilidade intersujeitos (importante para ML)

In [11]:
degree_stats = (
    df_degree_long
    .groupby(["node", "condition"])["degree"]
    .agg(["mean", "std"])
    .reset_index()
)

In [12]:
fig = px.bar(
    degree_stats,
    x="node",
    y="std",
    color="condition",
    barmode="group",
    title="Variabilidade Inter-Sujeitos do Grau por Nó"
)
fig.update_layout(xaxis_tickangle=45)
fig.show()

### Heatmap sujeito × nó (padrões globais)

In [13]:
df_degree_matrix = df_node[degree_cols].copy()
df_degree_matrix.index = df_node["subject_id"]

In [14]:
fig = px.imshow(
    df_degree_matrix,
    aspect="auto",
    title="Heatmap do Grau por Nó (Sujeito × Nó)",
    color_continuous_scale="Viridis"
)
fig.show()

df_degree_long.groupby(["node", "condition"])["degree"].describe()


In [15]:
df_degree_long.groupby(["node", "condition"])["degree"].describe()

count      mean       std       min       25%       50%  \
node       condition                                                            
degree_AF3 EC          54.0  0.489010  0.095537  0.253507  0.423366  0.497458   
           EO          54.0  0.459453  0.106417  0.214692  0.394031  0.464883   
degree_AF4 EC          54.0  0.478107  0.102660  0.226937  0.407934  0.491354   
           EO          54.0  0.452273  0.112237  0.174317  0.401147  0.456519   
degree_AF8 EC          54.0  0.507736  0.104752  0.282966  0.413929  0.512330   
...                     ...       ...       ...       ...       ...       ...   
degree_POz EO          54.0  0.615431  0.071667  0.460295  0.569075  0.618665   
degree_Pz  EC          54.0  0.614194  0.058658  0.438131  0.585056  0.613807   
           EO          54.0  0.610584  0.072891  0.462315  0.571018  0.606029   
degree_TP8 EC          53.0  0.605610  0.086257  0.427296  0.572767  0.606211   
           EO          53.0  0.581615  0.107493  0.347475  0.509227  0.585465   

                           75%       max  
node       condition                      
degree_AF3 EC         0.545967  0.705482  
           EO         0.515088  0.691493  
degree_AF4 EC         0.551663  0.714794  
           EO         0.501443  0.733435  
degree_AF8 EC         0.586930  0.771725  
...                        ...       ...  
degree_POz EO         0.657666  0.822589  
degree_Pz  EC         0.649788  0.759282  
           EO         0.652269  0.798492  
degree_TP8 EC         0.667353  0.777550  
           EO         0.645644  0.805491  

[106 rows x 8 columns]